### Homework 5 (10pt): Question search engine

Remeber Week01, where you used GloVe embeddings to find related questions? That was... cute. Now, it's time to really solve this task using context-aware embeddings.

__Warning:__ this task assumes you have seen `practice06.ipynb` [notebook](https://github.com/anton-selitskiy/RIT_LLM/blob/main/Week06_bert/practice06.ipynb)

This assignmend is inspired by this [notebook](https://github.com/yandexdataschool/nlp_course/blob/2024/week05_transfer/homework.ipynb)

In [1]:
# !pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

### Load data and model

In [2]:
qqp = datasets.load_dataset('SetFit/qqp')
print('\n')
print("Sample[0]:", qqp['train'][0])
print("Sample[3]:", qqp['train'][3])

Repo card metadata block was not found. Setting CardData to empty.




Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [4]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

### Tokenize the data

In [5]:
MAX_LENGTH = 128
def preprocess_function(examples):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['label'] = examples['label']
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

In [6]:
print(repr(qqp_preprocessed['train'][0]['input_ids'])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


In [7]:
print(tokenizer.decode(qqp_preprocessed['train'][0]['input_ids']))

[CLS] How is the life of a math student? Could you describe your own experiences? [SEP] Which level of prepration is enough for the exam jlpt5? [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


### Task 1: evaluation (3 point)

We randomly chose a model trained on QQP - but is it any good?

One way to assess this is by measuring validation accuracy, which you will implement next.

Here’s the interface to help you get started:

In [9]:
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator, num_workers=2
)

In [8]:
for batch in val_loader:
    break  # here will be your training code later. For now, it reads one batch
print("Sample batch:", batch)

with torch.no_grad():
  predicted = model(
      input_ids=batch['input_ids'],
      attention_mask=batch['attention_mask'],
      token_type_ids=batch['token_type_ids']
  )

print('\nPrediction (probs):', torch.softmax(predicted.logits, dim=1).data.numpy())

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Users/log/Documents/cs539/RIT_LLM/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/log/Documents/cs539/RIT_LLM/.venv/lib/python3.9/site-packages

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

__Your task__ is to measure the validation accuracy of your model.
Doing so naively may take several hours. Please make sure you use the following optimizations:

- run the model on GPU with no_grad
- using batch size larger than 1
- use optimize data loader with num_workers > 1
- (optional) use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [10]:
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=16, shuffle=False, collate_fn=transformers.default_data_collator, num_workers=4
)

In [12]:
from tqdm.notebook import tqdm

In [ ]:
# <A lot of YOUR CODE HERE>
# ...

with torch.inference_mode():
    correct = 0
    total = 0
    for batch in val_loader:
        predicted = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            token_type_ids=batch['token_type_ids']
        )
        probs = torch.softmax(predicted.logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        
        correct += (preds == batch['labels']).sum().item()
        total += batch['labels'].size(0)
accuracy = correct / total
print(f'Validation Accuracy: {accuracy:.4f}')


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Users/log/Documents/cs539/R

Validation Accuracy: 0.9084


In [17]:
assert 0.9 < accuracy < 0.91

### Task 2: train the model (5 points)

Fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base), but you can choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually (as we did in class) or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
model_name = "microsoft/deberta-v3-base"
tokenizer = transformers.DebertaV2Tokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
epochs = 3
print(device)


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


cuda


In [6]:
# tokenize data
MAX_LENGTH = 128
def preprocess_function(examples):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['label'] = examples['label']
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True)

In [6]:
print(repr(qqp_preprocessed['train'][0]['input_ids'])[:100], "...")
print(tokenizer.decode(qqp_preprocessed['train'][0]['input_ids']))

[1, 577, 269, 262, 432, 265, 266, 5291, 1234, 302, 5047, 274, 3443, 290, 451, 2056, 302, 2, 2597, 67 ...
[CLS] How is the life of a math student? Could you describe your own experiences?[SEP] Which level of prepration is enough for the exam jlpt5?[SEP][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD]


In [7]:
train_set = qqp_preprocessed['train']
train_loader = torch.utils.data.DataLoader(
    train_set, batch_size=16, shuffle=False, collate_fn=transformers.default_data_collator, num_workers=4
)

In [11]:
from tqdm.notebook import tqdm

In [13]:
# training loop
for epoch in range(epochs):
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            token_type_ids=batch['token_type_ids'].to(device),
            labels=batch['labels'].to(device)
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch} Loss: {loss.item()}')


Exception ignored in: <function tqdm.__del__ at 0x0000018A34DCEB60>
Traceback (most recent call last):
  File "c:\Users\Logan\Documents\CODE\RIT_LLM\.venv\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "c:\Users\Logan\Documents\CODE\RIT_LLM\.venv\Lib\site-packages\tqdm\notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


Epoch 0 Loss: 0.25641945004463196
Epoch 1 Loss: 0.036103811115026474
Epoch 2 Loss: 0.02760186791419983


In [ ]:
# validation
model.eval()
with torch.inference_mode():
    correct = 0
    total = 0
    for batch in val_loader:
        predicted = model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            token_type_ids=batch['token_type_ids'].to(device)
        )
        probs = torch.softmax(predicted.logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        
        correct += (preds == batch['labels'].to(device)).sum().item()
        total += batch['labels'].size(0)
accuracy = correct / total

In [22]:
assert 0.9 < accuracy < 0.95

In [ ]:
torch.save(model.state_dict(), 'qqp_model1.pth')

In [11]:
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.load_state_dict(torch.load('qqp_model.pth'))
# validation
model.eval()
with torch.inference_mode():
    correct = 0
    total = 0
    for batch in val_loader:
        predicted = model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            token_type_ids=batch['token_type_ids'].to(device)
        )
        probs = torch.softmax(predicted.logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        
        correct += (preds == batch['labels'].to(device)).sum().item()
        total += batch['labels'].size(0)
accuracy = correct / total

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
print(f'Validation Accuracy: {accuracy:.4f}')

Validation Accuracy: 0.9176


### Task 3: try the full pipeline (2 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 3 examples.

In [48]:
@torch.inference_mode()
def get_question_features(question, pipeline):
    return pipeline(question)

@torch.inference_mode()
def get_train_features(train_set, pipeline):
    return pipeline(train_set['text1'])

@torch.inference_mode()
def get_similarities(question_features, train_features):
    question_features = torch.tensor(question_features).to(device)
    train_features = torch.tensor(train_features).to(device)
    return F.cosine_similarity( train_features, question_features.unsqueeze(0).expand_as(train_features), dim=1)

In [52]:
from transformers import pipeline

In [36]:
pipe = pipeline('feature-extraction', model=model, tokenizer=tokenizer, device=device)

Device set to use cuda


In [ ]:
train_features = get_train_features(train_set, pipe)

KeyboardInterrupt: 

The code block above displays an error output, but that is because I stopped it early after it began to run again. The block above took 80 minutes to run and I already got the results from it into memory.

In [49]:
def get_duplicates(similarities):
    return similarities.argsort(descending=True)[:5]

In [50]:
question1 = "How can I improve my English?"

question1_features = get_question_features(question1, pipe)
similarities1 = get_similarities(question1_features, train_features)
duplicates1 = get_duplicates(similarities1) # returs indices for top 5 duplicates


In [51]:
question2 = "How many licks does it take to get to the center of a tootsie pop?"
question2_features = get_question_features(question2, pipe)
similarities2 = get_similarities(question2_features, train_features)
duplicates2 = get_duplicates(similarities2)
print(duplicates2)

question3 = "What is the capital of France?"
question3_features = get_question_features(question3, pipe)
similarities3 = get_similarities(question3_features, train_features)
duplicates3 = get_duplicates(similarities3)
print(duplicates3)

print("Question 1:", question1)
print("Duplicates 1:", end='\t')
for i in duplicates1:
    print("\t", train_set[i]['text1'])
print()
print("Question 2:", question2)
print("Duplicates 2:", end='\t')
for i in duplicates2:
    print("\t", train_set[i]['text1'])
print()
print("Question 3:", question3)
print("Duplicates 3:", end='\t')
for i in duplicates3:
    print("\t", train_set[i]['text1'])

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


tensor([[1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0]], device='cuda:0')
tensor([[1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0]], device='cuda:0')
Question 1: How can I improve my English?
Duplicates 1:		 ['How do I control my horny emotions?', 'How is the life of a math student? Could you describe your own experiences?']
	 ['How do I control my horny emotions?', 'How is the life of a math student? Could you describe your own experiences?']
	 ['How do I control my horny emotions?', 'How is the life of a math student? Could you describe your own experiences?']
	 ['How do I control my horny emotions?', 'How is the life of a math student? Could you describe your own experiences?']
	 ['How do I control my horny emotions?', 'How is the life of a math student? Could you describe your own experiences?']

Question 2: How many licks does it take to get to the center of a tootsie pop?
Duplicates 2:		 ['How do I control my horny emotions?', 'How is

I think that I tried to use the pipeline in the wrong way, so the cosine similarity returned the same output for every question. 